In [ ]:
import FIRM.base.operators.implications as implications
import FIRM.base.operators.tnorms as tnorms
import FIRM.base.fuzzy_data as fuzzy_data
from FIRM.methods.AARFI import AARFI_F
import csv
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
import FIRM.base.ct_fuzzy_rule as fuzzy_rule
import math
from FIRM.base.ct_set_fuzzy_rules import SetFuzzyRules

In [ ]:
def _tokens_from_group(group):
    """Return a list of 'Var_Label' tokens from frozenset/set/list/tuple or string."""
    if group is None or (isinstance(group, float) and math.isnan(group)):
        return []
    if isinstance(group, (set, frozenset, list, tuple)):
        return [str(x).strip() for x in group if str(x).strip()]
    if isinstance(group, str):
        s = group.strip()
        if not s:
            return []
        if s.startswith("(") and s.endswith(")"):
            s = s[1:-1].strip()
        return [t.strip() for t in s.split(",")] if "," in s else [s]
    return [str(group).strip()]


def _build_var_and_label_maps(dataset_columns, fuzzy_dataset):
    """
    Returns:
      var_to_idx: {var_name -> position i}
      label_maps: {var_name -> {label_string -> label_index}}
    Labels come from fuzzy_dataset.fv_list[i].get_labels()
    """
    cols_list = list(dataset_columns.tolist()) if hasattr(dataset_columns, "tolist") else list(dataset_columns)
    var_to_idx = {name: i for i, name in enumerate(cols_list)}

    label_maps = {}
    for var_name, i in var_to_idx.items():
        # CALL the method to get the actual label list
        labels = list(fuzzy_dataset.fv_list[i].get_labels)
        # keep original strings; also add a case-insensitive shim
        per_var = {str(lab): j for j, lab in enumerate(labels)}
        label_maps[var_name] = per_var
    return var_to_idx, label_maps


def _resolve_label_idx(var: str, lab: str, label_maps: dict) -> int:
    """
    Map a label string to its index using label_maps[var].
    Tries exact, then case-insensitive exact, then first-letter (if unique).
    """
    per_var = label_maps[var]
    if lab in per_var:
        return per_var[lab]
    # case-insensitive exact
    for k in per_var.keys():
        if k.lower() == lab.lower():
            return per_var[k]
    # First-letter fallback, only if unique
    fl = lab[:1].lower()
    candidates = [name for name in per_var.keys() if name[:1].lower() == fl]
    if len(candidates) == 1:
        return per_var[candidates[0]]
    raise KeyError(f"Unknown label {lab!r} for variable {var!r}. Known labels: {list(per_var.keys())}")


def _pair_from_token(token: str, var_to_idx: dict, label_maps: dict) -> tuple[int, int]:
    """Map 'Var_Label' -> (var_idx, label_idx) using dataset column order + fuzzy_dataset labels."""
    if "_" not in token:
        raise ValueError(f"Expected 'Var_Label', got: {token!r}")
    var, lab = token.rsplit("_", 1)  # last underscore splits var from label
    var = var.strip()
    lab = lab.strip()
    if var not in var_to_idx:
        raise KeyError(f"Variable {var!r} not found in dataset columns: {list(var_to_idx.keys())}")
    var_idx = var_to_idx[var]
    lab_idx = _resolve_label_idx(var, lab, label_maps)
    return (var_idx, lab_idx)


def _group_to_pairs(group, var_to_idx, label_maps):
    tokens = _tokens_from_group(group)
    pairs = [_pair_from_token(t, var_to_idx, label_maps) for t in tokens]
    # Deterministic order (since sets/frozensets are unordered)
    pairs.sort(key=lambda x: x[0])
    return pairs


def _pick_consequent(con_pairs):
    """
    Choose one consequent tuple from con_pairs.
    - If exactly one, use it.
    - If multiple, pick the one with the highest variable index (you can change to lowest).
    """
    if not con_pairs:
        raise ValueError("Rule has no consequent.")
    if len(con_pairs) == 1:
        return con_pairs[0]
    # pick by highest var index (change to min if you prefer)
    return max(con_pairs, key=lambda x: x[0])


def df_to_crfuzzyrules(df: pd.DataFrame, dataset_columns, fuzzy_dataset):
    """
    Build CRFuzzyRule objects from df['antecedents'] and df['consequents'].
    Each CRFuzzyRule receives [*antecedents, consequent] (consequent appended last).
    Returns (rules, consequents_as_singleton_lists) to keep backward compatibility.
    """
    var_to_idx, label_maps = _build_var_and_label_maps(dataset_columns, fuzzy_dataset)

    rules, consequents = [], []
    for _, row in df.iterrows():
        ant_pairs = _group_to_pairs(row["antecedents"], var_to_idx, label_maps)
        con_pairs = _group_to_pairs(row["consequents"], var_to_idx, label_maps)
        c = _pick_consequent(con_pairs)
        combined = ant_pairs + [c]  # consequent at the end
        rules.append(fuzzy_rule.CRFuzzyRule(combined))
        consequents.append([c])  # keep a reference to the chosen consequent
    return rules, consequents

In [ ]:
impl_operators = [implications.ImplicationsExamples.get_fuzzy_implication(implications.ImplicationsExamples.IGNORE),
                  implications.ImplicationsExamples.get_fuzzy_implication(implications.ImplicationsExamples.LUKASIEWICZ),
                  lambda x, y: implications.ImplicationsExamples.get_fuzzy_implication(implications.ImplicationsExamples.KSS)(x, y, float(-10)),
                  lambda x, y: 1 - x + x * (y**0.01)]
tnorms_operators = [tnorms.TnormsExamples.get_tnorm(tnorms.TnormsExamples.PRODUCT),
                    tnorms.TnormsExamples.get_tnorm(tnorms.TnormsExamples.LUKASIEWICZ),
                    lambda x, y: tnorms.TnormsExamples.get_tnorm(tnorms.TnormsExamples.SCHWEIZER_SKLAR)(x, y,float(-10)),
                    lambda x, y: np.maximum(x + y - 1, 0)]

p = -10.0  # Schweizer–Sklar parameter

k    = lambda x: np.exp((np.power(x, p) - 1.0) / p)
kinv = lambda x: np.power(1.0 + p * np.log(x), 1.0 / p)

F_operators = [
    lambda x, y: x * y,                                  # product
    lambda x, y: np.minimum(x, y),                       # Gödel/min
    lambda x, y: kinv(k(x) * np.minimum(k(y) / np.clip(x, 1e-12, None), 1.0)),
    lambda x, y: x * np.power(y, 0.01),                  # x * y^0.01
]

In [ ]:
def process_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Integer dtypes -> float64
    - For object/string/categorical columns:
        * If unique non-null values <= 15: keep as object
        * If unique non-null values > 15: keep top 15, others -> "Unknown"
    - Other dtypes left unchanged
    """
    out = df.copy()

    for col in out.columns:
        s = out[col]

        # 1) Integers -> float64
        if pd.api.types.is_integer_dtype(s):
            out[col] = s.astype("float64")
            continue

        # 2) Non-numeric categoricals/strings
        is_cat_like = (
            pd.api.types.is_object_dtype(s)
            or isinstance(s.dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(s)
        )
        if is_cat_like:
            n_unique = s.nunique(dropna=True)
            if n_unique > 10:
                top15 = s.value_counts(dropna=True).index[:10]
                out[col] = s.where(s.isna() | s.isin(top15), "Unknown").astype("object")
            else:
                out[col] = s.astype("object")

    return out

In [ ]:
# 'online_news.csv','global_house.csv'
datasets = ['iris.csv','wdbc.csv','vehicle.csv','abalone.csv','magic.csv','online_news.csv','global_house.csv']

In [ ]:
import csv
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

n_datasets = len(datasets)
n_operators = len(impl_operators)

# results[op_idx][dat_idx] -> list of CRFuzzyRule (last op_idx = crisp)
results = [[[] for _ in range(n_datasets)] for _ in range(n_operators + 1)]

with open('results.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Operator', 'Dataset', 'Num Rules', 'Coverage', 'Support', 'Confidence'])

    # -------- fuzzy operators --------
    for idx_op in range(n_operators):
        print('Operator:', idx_op)
        I = impl_operators[idx_op]
        T = tnorms_operators[idx_op]
        F = F_operators[idx_op]

        for idx_dat in range(n_datasets):
            name_dataset = datasets[idx_dat]
            print('Dataset:', name_dataset)

            dataset = process_df(pd.read_csv('../assets/' + name_dataset, sep=','))
            # ints -> float
            int_cols = dataset.select_dtypes(include=['int']).columns
            if len(int_cols):
                dataset[int_cols] = dataset[int_cols].astype(float)

            # NOTE: first arg is the shape specifier
            fuzzy_dataset = fuzzy_data.FuzzyDataQuantiles(name_dataset, dataset, 3, ['L', 'M', 'H'])

            rules, measures = AARFI_F(dataset, fuzzy_dataset, F=F, T=T, I=I,
                                      min_cov=0.3, min_supp=0.3, min_conf=0.8,
                                      max_feat=3, verbose=True)

            # Extract measures
            num_rules = int(len(measures['n_antecedents']))
            coverage  = float(np.mean(measures['coverage']))   if num_rules else np.nan
            support   = float(np.mean(measures['support']))    if num_rules else np.nan
            confidence= float(np.mean(measures['confidence'])) if num_rules else np.nan

            print('  num rules:', num_rules)
            print('  coverage :', coverage)
            print('  support  :', support)
            print('  confidence:', confidence)

            # Write CSV row (inside the with-block!)
            writer.writerow([f'OP{idx_op}', name_dataset, num_rules, coverage, support, confidence])

            # Store the rule list
            results[idx_op][idx_dat] = rules.rule_list

    # -------- crisp case (now INSIDE the with-block) --------
    print('Crisp Case ----')
    for idx_dat in range(n_datasets):
        name_dataset = datasets[idx_dat]
        print('Dataset:', name_dataset)

        dataset = process_df(pd.read_csv('../assets/' + name_dataset, sep=','))
        # ints -> float
        int_cols = dataset.select_dtypes(include=['int']).columns
        if len(int_cols):
            dataset[int_cols] = dataset[int_cols].astype(float)

        fuzzy_dataset = fuzzy_data.FuzzyDataQuantiles(name_dataset, dataset, 3, ['L', 'M', 'H'])

        data = dataset.copy()
        for i in range(len(fuzzy_dataset.fv_list)):
            data[dataset.columns[i]] = dataset[dataset.columns[i]].map(
                lambda x: fuzzy_dataset.fv_list[i].eval_max_fuzzy_set(x)
            )

        df_encoded = pd.get_dummies(data, columns=data.columns)
        df_freq = apriori(df_encoded, min_support=0.3, use_colnames=True, verbose=1,max_len=4,low_memory=True)
        df_ar   = association_rules(df_freq, metric="confidence", min_threshold=0.8)

        # Keep only rules with <=3 items in antecedent and <=1 in consequent
        df_rules_filtered = df_ar[
            (df_ar['antecedents'].apply(len) <= 3) &
            (df_ar['consequents'].apply(len) <= 1)
        ].reset_index(drop=True)

        rules_sorted = df_rules_filtered.sort_values(by="confidence", ascending=False).reset_index(drop=True)
        num_rules = int(len(rules_sorted))

        # Crisp “coverage” = antecedent support in mlxtend
        coverage  = float(rules_sorted['antecedent support'].mean()) if num_rules else np.nan
        support   = float(rules_sorted['support'].mean())            if num_rules else np.nan
        confidence= float(rules_sorted['confidence'].mean())         if num_rules else np.nan

        print('  num rules:', num_rules)
        print('  coverage :', coverage)
        print('  support  :', support)
        print('  confidence:', confidence)

        # Write CSV row (still inside the with-block!)
        writer.writerow(['CRISP', name_dataset, num_rules, coverage, support, confidence])

        # Convert crisp rules to CRFuzzyRule objects and store
        dataset_columns = dataset.columns.str.strip()
        rules_crisp, _ = df_to_crfuzzyrules(rules_sorted, dataset_columns, fuzzy_dataset)
        results[n_operators][idx_dat] = rules_crisp  # last index reserved for crisp

print("Results have been saved to 'results.csv'")

In [ ]:
M = np.ones((n_operators+1, n_operators+1))
for i in range(0, n_operators+1):
    for j in range(i + 1, n_operators+1):
        diss_perc = []
        for d in range(0, n_datasets):
            rules1 = SetFuzzyRules(results[i][d][:]) 
            rules2 = SetFuzzyRules(results[j][d][:])
            diss_perc = diss_perc + [rules1.jaccard_similarity(rules2)]
        M[i][j] = round(np.mean(diss_perc),3)

In [ ]:
M

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Your (possibly half-filled) matrix M and labels ---
labels = ["TP", "TM", "(TSSλ,ISSλ)", "(TLK,Ip)", "Crisp"]

M = np.asarray(M, dtype=float)  # ensure ndarray

# --- Symmetrize: take the upper triangle (incl. diag) and mirror it down ---
M_up   = np.triu(M)                               # keep upper triangle (incl. diag)
M_sym  = M_up + M_up.T - np.diag(np.diag(M_up))   # reflect to lower; keep original diag
np.fill_diagonal(M_sym, 1.0)                      # ensure diag = 1

# --- Build DataFrame and plot full heatmap (no mask) ---
df_full = pd.DataFrame(M_sym, index=labels, columns=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    df_full,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=0, vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8}
)
#plt.title("Mean Jaccard similarity between operators (including Crisp)")
plt.tight_layout()
plt.show()
